In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score,precision_recall_curve, auc, precision_score, recall_score, confusion_matrix

import joblib

In [16]:
# Load the dataset
df= pd.read_csv("credit_train.csv")

In [17]:
#preprocessing the data
df = df.dropna(subset=["Loan Status"])
df = df.drop(columns=["Loan ID", "Customer ID"])

#encoding categorical variables
df["Loan Status"] = df["Loan Status"].map({
    "Fully Paid": 1,
    "Charged Off": 0
})

df["Years in current job"] = df["Years in current job"].replace({
    "< 1 year": 0, "1 year": 1, "2 years": 2,
    "3 years": 3, "4 years": 4, "5 years": 5,
    "6 years": 6, "7 years": 7, "8 years": 8,
    "9 years": 9, "10+ years": 10
})

df = df.drop(columns=["Months since last delinquent"])

# Define features and target variable
X = df.drop("Loan Status", axis=1)
y = df["Loan Status"]

# Identify numeric and categorical columns
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])
 
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
 
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])


C:\Users\Sakthi\AppData\Local\Temp\ipykernel_8716\2931158334.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Years in current job"] = df["Years in current job"].replace({


In [18]:
# ── 5. Split ───────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")
print(f"Class balance — 0 (Charged Off): {(y_train==0).sum()} | 1 (Fully Paid): {(y_train==1).sum()}\n")

Train: 80000 rows | Test: 20000 rows
Class balance — 0 (Charged Off): 18111 | 1 (Fully Paid): 61889



In [19]:
# ── 6. Model with class_weight='balanced' ──────────────────────────────────
# WHY THIS FIXES PREDICT-ALL-1:
#   SMOTE created so many synthetic class-0 samples that the model
#   became confused and defaulted to always predicting 1.
#   class_weight='balanced' instead tells RandomForest internally to
#   penalise misclassifying the minority class (0) more heavily —
#   clean, effective, no synthetic data needed.
 
rf = RandomForestClassifier(
    n_estimators=100,        # 100 trees keeps file size small
    max_depth=12,            # enough depth to learn real patterns
    min_samples_leaf=10,     # prevents overfitting to majority class
    max_features="sqrt",     # standard for classification
    class_weight="balanced", # THE core fix — replaces SMOTE
    random_state=42,
    n_jobs=-1
)
 
full_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", rf)
])
 
print("Training... (this takes ~30-60 seconds)")
full_pipeline.fit(X_train, y_train)
print("Done.\n")

Training... (this takes ~30-60 seconds)
Done.



In [20]:
# ── 7. Default threshold results ───────────────────────────────────────────
y_prob = full_pipeline.predict_proba(X_test)[:, 1]
y_pred = full_pipeline.predict(X_test)
 
print("=" * 58)
print("RESULTS AT DEFAULT THRESHOLD (0.50)")
print("=" * 58)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}\n")
print(classification_report(
    y_test, y_pred,
    target_names=["Charged Off (0)", "Fully Paid (1)"]
))
 


RESULTS AT DEFAULT THRESHOLD (0.50)
Accuracy : 0.7202
ROC-AUC  : 0.7583

                 precision    recall  f1-score   support

Charged Off (0)       0.42      0.58      0.48      4528
 Fully Paid (1)       0.86      0.76      0.81     15472

       accuracy                           0.72     20000
      macro avg       0.64      0.67      0.65     20000
   weighted avg       0.76      0.72      0.73     20000



In [21]:


# ── 9. Save ────────────────────────────────────────────────────────────────

joblib.dump(full_pipeline, "model.joblib",   compress=9)



['model.joblib']

In [22]:
print(df.columns)    
print(X.columns.tolist())

Index(['Loan Status', 'Current Loan Amount', 'Term', 'Credit Score',
       'Annual Income', 'Years in current job', 'Home Ownership', 'Purpose',
       'Monthly Debt', 'Years of Credit History', 'Number of Open Accounts',
       'Number of Credit Problems', 'Current Credit Balance',
       'Maximum Open Credit', 'Bankruptcies', 'Tax Liens'],
      dtype='object')
['Current Loan Amount', 'Term', 'Credit Score', 'Annual Income', 'Years in current job', 'Home Ownership', 'Purpose', 'Monthly Debt', 'Years of Credit History', 'Number of Open Accounts', 'Number of Credit Problems', 'Current Credit Balance', 'Maximum Open Credit', 'Bankruptcies', 'Tax Liens']
